In [10]:
!pip install pandas

In [6]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [8]:
data = pd.read_csv("/content/Titanic-Dataset.csv")

print("Dataset shape:", data.shape)
data.head()


Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [11]:
print(data.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [12]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

target = "Survived"

X = data[features]
y = data[target]

print(X.head())
print(y.head())

   Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0       3    male  22.0      1      0   7.2500        S
1       1  female  38.0      1      0  71.2833        C
2       3  female  26.0      0      0   7.9250        S
3       1  female  35.0      1      0  53.1000        S
4       3    male  35.0      0      0   8.0500        S
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 712
Testing records: 179


In [14]:
numerical_only = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

baseline_train = X_train[numerical_only].copy()
baseline_test = X_test[numerical_only].copy()

train_valid_rows = baseline_train.dropna().index
test_valid_rows = baseline_test.dropna().index

baseline_train = baseline_train.loc[train_valid_rows]
baseline_test = baseline_test.loc[test_valid_rows]

y_train_baseline = y_train.loc[train_valid_rows]
y_test_baseline = y_test.loc[test_valid_rows]

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(baseline_train, y_train_baseline)

baseline_predictions = baseline_model.predict(baseline_test)

baseline_accuracy = accuracy_score(
    y_test_baseline,
    baseline_predictions
)

print("Baseline accuracy:", baseline_accuracy)
print("Training records used:", len(baseline_train))
print("Testing records used:", len(baseline_test))

Baseline accuracy: 0.6762589928057554
Training records used: 575
Testing records used: 139


In [15]:
numerical_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Sex",
    "Embarked"
]

In [16]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [17]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [19]:
complete_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [20]:
complete_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [21]:
processed_predictions = complete_pipeline.predict(X_test)

processed_accuracy = accuracy_score(
    y_test,
    processed_predictions
)

print("Accuracy after preprocessing:", processed_accuracy)

Accuracy after preprocessing: 0.8044692737430168


In [22]:
print(
    classification_report(
        y_test,
        processed_predictions
    )
)

              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [23]:
comparison = pd.DataFrame(
    {
        "Approach": [
            "Before proper preprocessing",
            "After preprocessing pipeline"
        ],
        "Accuracy": [
            baseline_accuracy,
            processed_accuracy
        ],
        "Training records used": [
            len(baseline_train),
            len(X_train)
        ],
        "Testing records used": [
            len(baseline_test),
            len(X_test)
        ]
    }
)

print(comparison)

                       Approach  Accuracy  Training records used  \
0   Before proper preprocessing  0.676259                    575   
1  After preprocessing pipeline  0.804469                    712   

   Testing records used  
0                   139  
1                   179  


In [24]:
X_train_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

In [25]:
feature_names = complete_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print(feature_names)

['numerical__Pclass' 'numerical__Age' 'numerical__SibSp'
 'numerical__Parch' 'numerical__Fare' 'categorical__Sex_female'
 'categorical__Sex_male' 'categorical__Embarked_C'
 'categorical__Embarked_Q' 'categorical__Embarked_S']


In [26]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [27]:
processed_train_df["Survived"] = y_train.values
processed_test_df["Survived"] = y_test.values

In [28]:
processed_train_df.head

<bound method NDFrame.head of      numerical__Pclass  numerical__Age  numerical__SibSp  numerical__Parch  \
692           0.829568       -0.081135         -0.465084         -0.466183   
481          -0.370945       -0.081135         -0.465084         -0.466183   
527          -1.571457       -0.081135         -0.465084         -0.466183   
855           0.829568       -0.887827         -0.465084          0.727782   
801          -0.370945        0.110934          0.478335          0.727782   
..                 ...             ...               ...               ...   
359           0.829568       -0.081135         -0.465084         -0.466183   
258          -1.571457        0.418245         -0.465084         -0.466183   
736           0.829568        1.417007          0.478335          3.115713   
462          -1.571457        1.340179         -0.465084         -0.466183   
507          -1.571457       -0.081135         -0.465084         -0.466183   

     numerical__Fare  categorical__Sex_female  categorical__Sex_male  \
692         0.513812                      0.0                    1.0   
481        -0.662563                      0.0                    1.0   
527         3.955399                      0.0                    1.0   
855        -0.467874                      1.0                    0.0   
801        -0.115977                      1.0                    0.0   
..               ...                      ...                    ...   
359        -0.498500                      1.0                    0.0   
258        10.005329                      1.0                    0.0   
736         0.053205                      1.0                    0.0   
462         0.139097                      0.0                    1.0   
507        -0.109730                      0.0                    1.0   

     categorical__Embarked_C  categorical__Embarked_Q  \
692                      0.0                      0.0   
481                      0.0                      0.0   
527                      0.0                      0.0   
855                      0.0                      0.0   
801                      0.0                      0.0   
..                       ...                      ...   
359                      0.0                      1.0   
258                      1.0                      0.0   
736                      0.0                      0.0   
462                      0.0                      0.0   
507                      0.0                      0.0   

     categorical__Embarked_S  Survived  
692                      1.0         1  
481                      1.0         0  
527                      1.0         0  
855                      1.0         1  
801                      1.0         1  
..                       ...       ...  
359                      0.0         1  
258                      0.0         1  
736                      1.0         0  
462                      1.0         0  
507                      1.0         1  

[712 rows x 11 columns]>

In [29]:
processed_train_df.to_csv(
    "titanic_processed_train.csv",
    index=False
)

processed_test_df.to_csv(
    "titanic_processed_test.csv",
    index=False
)

In [30]:
from google.colab import files

files.download("titanic_processed_train.csv")
files.download("titanic_processed_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
joblib.dump(
    complete_pipeline,
    "titanic_preprocessing_model_pipeline.pkl"
)

['titanic_preprocessing_model_pipeline.pkl']

In [32]:
files.download(
    "titanic_preprocessing_model_pipeline.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
loaded_pipeline = joblib.load(
    "titanic_preprocessing_model_pipeline.pkl"
)

In [34]:
new_passenger = pd.DataFrame(
    {
        "Pclass": [3],
        "Sex": ["female"],
        "Age": [25],
        "SibSp": [0],
        "Parch": [0],
        "Fare": [7.25],
        "Embarked": ["S"]
    }
)

In [35]:
prediction = loaded_pipeline.predict(new_passenger)

if prediction[0] == 1:
    print("Predicted result: Survived")
else:
    print("Predicted result: Did not survive")

Predicted result: Survived
